In [ ]:
from openai import OpenAI
from bs4 import BeautifulSoup
import requests
import json

In [ ]:
baseUrl = "http://localhost:11434/v1"
api_key="ollama"
openai = OpenAI(base_url=baseUrl,api_key=api_key)

In [ ]:
#fetching Html data 
url = "https://medium.com/@RobuRishabh/introduction-to-machine-learning-555b0f1b62f5"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def fetch(url):
    try:
        r = requests.get(url + "?format=json", headers=HEADERS, timeout=10)
        data = r.json()
        post = data["payload"]["value"]["post"]
        paragraphs = post["content"]["bodyModel"]["paragraphs"]
        return {
            "title": post.get("title", ""),
            "author": post.get("creator", {}).get("name", ""),
            "text": "\n".join(p["text"] for p in paragraphs if p.get("text"))
        }
    except Exception:
        pass

    r = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    article = soup.find("article")
    return {
        "text": article.get_text("\n") if article else "\n".join(p.get_text() for p in soup.find_all("p"))
    }


In [ ]:
# set up environment
blogText = fetch(url)["text"]
systemPrompt = """
You are a blog summarizer whose task is to create clear, structured summaries of blog articles for beginner readers.

Your output must always be in three parts, labeled exactly as follows:

1. Introduction: A brief explanation…
2. Summary: A concise but complete summary…
3. Key Points: A list of the most important takeaways…

Write in simple, clear language suitable for beginners.
"""

userPrompt = f"""
Please provide a detailed summary for the blog text below, following the three‑part format defined in the system instructions:

Blog Text: {blogText}
"""
messages = [
    {
        "role" : "system", 
        "content": systemPrompt
        },
    {   
        "role":"user",
        "content" : userPrompt
        }
]

response = openai.chat.completions.create(
    model = 'llama3.2',
    messages = messages,
    stream=True
)
for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)